In [1]:
import numpy as np
import pretty_midi
from pretty_midi import PrettyMIDI
from scipy.stats import entropy
from collections import defaultdict
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import os
os.chdir('..')
from src import cli
print(os.getcwd())
os.chdir('./notebooks')


c:\Users\giova\Nextcloud\3. Semeseter\DL4AD\dequant


In [2]:
# Drum category definition
DRUM_CATEGORIES = {
    "Kick": (36, 35),
    "Snare": (38, 37, 39, 40),
    "Floor Tom": (41, 43),
    "Low Tom": (45, 47),
    "High Tom": (48, 50),
    "Closed Hi-Hat": (42, 44),
    "Open Hi-Hat": (46,),
    "Crash": (49, 52, 55, 57),
    "Ride": (51, 53, 59),
}

# Drum category assignment
pitch_to_canonical = {}
for category, pitches in DRUM_CATEGORIES.items():
    canonical = pitches[0]  # first pitch is canonical
    for p in pitches:
        pitch_to_canonical[p] = canonical

# Compute 16th note timestep length
def get_16th_duration(midi):
    """if isinstance(midi, PrettyMIDI):
            midi = midi
    else:
        try:
            midi = PrettyMIDI(midi)
        except Exception as e:
            raise ValueError(f"Failed to parse MIDI file {midi}: {e}")"""
        
    _, tempi = midi.get_tempo_changes()
    
    if len(tempi) == 0:
        bpm = 120.0

    if len(tempi) > 1:
        bpm = float(tempi[0])

    bpm = tempi[0]

    #bpm = round(extract_tempo(midi))
    return (60.0 / bpm) / 4.0

# Convert MIDI to grid representation
def midi_to_grid_dict(midi_path):
    midi = pretty_midi.PrettyMIDI(midi_path)
    step_length = get_16th_duration(midi)

    grid_notes = {}

    for instrument in midi.instruments:
        if not instrument.is_drum:
            continue

        for note in instrument.notes:
            if note.pitch not in pitch_to_canonical:
                continue

            canonical_pitch = pitch_to_canonical[note.pitch]

            # Determine nearest 16th timestep
            grid_index = int(round(note.start / step_length))
            grid_time = grid_index * step_length

            key = (grid_index, canonical_pitch)

            # Keep only highest velocity in same cell
            if key not in grid_notes:
                grid_notes[key] = {
                    "start": note.start,
                    "velocity": note.velocity,
                    "grid_time": grid_time
                }
            else:
                if note.velocity > grid_notes[key]["velocity"]:
                    grid_notes[key] = {
                        "start": note.start,
                        "velocity": note.velocity,
                        "grid_time": grid_time
                    }

    return grid_notes, step_length

# Match two files based on grid
def match_on_grid(dict_A, dict_B):
    matched_keys = set(dict_A.keys()).intersection(set(dict_B.keys()))

    A_start = []
    B_start = []
    A_vel = []
    B_vel = []
    grid_times = []

    for key in sorted(matched_keys):
        A_note = dict_A[key]
        B_note = dict_B[key]

        A_start.append(A_note["start"])
        B_start.append(B_note["start"])
        A_vel.append(A_note["velocity"])
        B_vel.append(B_note["velocity"])
        grid_times.append(A_note["grid_time"])  # same grid time

    return (
        np.array(A_start),
        np.array(B_start),
        np.array(A_vel),
        np.array(B_vel),
        np.array(grid_times)
    )

# MAE/MSE computation
def compute_timing_errors(A_start, B_start):
    diff_ms = (B_start - A_start) * 1000.0
    mae = np.mean(np.abs(diff_ms))
    mse = np.mean(diff_ms ** 2)
    return mae, mse

# KL divergence
def compute_kl(real_values, gen_values, bins=50, value_range=None):
    hist_real, bin_edges = np.histogram(
        real_values, bins=bins, range=value_range
    )
    hist_gen, _ = np.histogram(
        gen_values, bins=bin_edges
    )

    p = hist_real.astype(np.float64)
    q = hist_gen.astype(np.float64)

    eps = 1e-8
    p += eps
    q += eps

    p /= p.sum()
    q /= q.sum()

    return entropy(p, q)

# Full evaluation
def evaluate(H_path, Q_path, G_path):

    # Convert to grid dicts
    H_dict, _ = midi_to_grid_dict(H_path)
    Q_dict, _ = midi_to_grid_dict(Q_path)
    G_dict, _ = midi_to_grid_dict(G_path)

    # Match H and G for MAE/MSE
    H_start, G_start, H_vel, G_vel, grid_times = \
        match_on_grid(H_dict, G_dict)

    timing_mae, timing_mse = compute_timing_errors(H_start, G_start)

    # Match H and Q
    H_start_Q, Q_start, _, _, grid_times_Q = \
        match_on_grid(H_dict, Q_dict)

    # Match G and Q
    G_start_Q, Q_start_G, _, _, grid_times_G = \
        match_on_grid(G_dict, Q_dict)

    # Compute offsets
    real_offsets = (H_start_Q - Q_start) * 1000.0
    gen_offsets  = (G_start_Q - Q_start_G) * 1000.0

    timing_kl = compute_kl(
        real_offsets,
        gen_offsets,
        bins=50,
        value_range=(-100, 100)
    )

    velocity_kl = compute_kl(
        H_vel,
        G_vel,
        bins=32,
        value_range=(0, 127)
    )

    """plt.hist(real_offsets, bins=50, alpha=0.5, label="Real", density=True)
    plt.hist(gen_offsets, bins=50, alpha=0.5, label="Generated", density=True)
    plt.legend()
    plt.title("Timing Offset Distribution")
    plt.show()"""

    return {
        "Timing MAE (ms)": timing_mae,
        "Timing MSE (ms²)": timing_mse,
        "Timing KL": timing_kl,
        "Velocity KL": velocity_kl
    }


In [3]:
META_PATH = Path("../.data/tmp/egmd-meta.csv")
ORIGINAL_ROOT = Path("../.data/tmp/egmd-midi/e-gmd-v1.0.0")
QUANTIZED_ROOT = Path("../.data/tmp/test/quantized")
DEQUANTIZED_ROOT = Path("../.data/tmp/test/dequantized")

CHECKPOINT = Path("C:\\Users\\giova\\Nextcloud\\Shared\\Model Archive\\2026_01_30 Before Posenc\\cp_20260130112914.pt")  # change manually

# Index dataset
def load_test_sequences():
    df = pd.read_csv(META_PATH)

    # Filter test split
    df = df[df["split"] == "test"]

    # Keep only 4-4
    df = df[df["time_signature"] == "4-4"]

    # Keep only one kit to avoid 43 duplicates
    df = df[df["kit_name"] == "Acoustic Kit"]

    # Keep only beats and discard fills
    df = df[df["beat_type"] == "beat"]

    # Remove duplicate sequences (just in case)
    #df = df.drop_duplicates(subset=["id"])

    print(f"Using {len(df)} unique sequences")

    return df

# Batch-infere the model
def run_dequantize_on_dataset(df):

    for _, row in tqdm(df.iterrows(), total=len(df)):
        midi_rel = Path(row["midi_filename"])
        
        (QUANTIZED_ROOT / midi_rel.parent).mkdir(parents=True, exist_ok=True)
        (DEQUANTIZED_ROOT / midi_rel.parent).mkdir(parents=True, exist_ok=True)
        
        original_path = ORIGINAL_ROOT / midi_rel
        quantized_path = QUANTIZED_ROOT / midi_rel
        dequantized_path = DEQUANTIZED_ROOT / midi_rel

        cli.run_quantize(original_path, quantized_path)
        
        cli.run_dequantize(quantized_path, dequantized_path, CHECKPOINT)

def evaluate_full_test_set(df):

    results_list = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        midi_rel = Path(row["midi_filename"])

        H_path = ORIGINAL_ROOT / midi_rel
        Q_path = QUANTIZED_ROOT / midi_rel
        G_path = DEQUANTIZED_ROOT / midi_rel

        if not G_path.exists():
            print(f"Missing generated file: {G_path}")
            continue

        try:
            metrics = evaluate(H_path, Q_path, G_path)
            results_list.append(metrics)

        except Exception as e:
            print(f"Error processing {midi_rel}: {e}")
            continue

    # Convert to arrays
    mae_values = np.array([r["Timing MAE (ms)"] for r in results_list])
    mse_values = np.array([r["Timing MSE (ms²)"] for r in results_list])
    timing_kl_values = np.array([r["Timing KL"] for r in results_list])
    velocity_kl_values = np.array([r["Velocity KL"] for r in results_list])

    print("\n==============================")
    print("FINAL TEST SET RESULTS")
    print("==============================")

    print(f"Number of evaluated sequences: {len(results_list)}")

    print("\nTiming MAE (ms):")
    print(f"Mean: {mae_values.mean():.3f}")
    print(f"Std:  {mae_values.std():.3f}")

    print("\nTiming MSE (ms²):")
    print(f"Mean: {mse_values.mean():.3f}")
    print(f"Std:  {mse_values.std():.3f}")

    print("\nTiming KL:")
    print(f"Mean: {timing_kl_values.mean():.3f}")
    print(f"Std:  {timing_kl_values.std():.3f}")

    print("\nVelocity KL:")
    print(f"Mean: {velocity_kl_values.mean():.3f}")
    print(f"Std:  {velocity_kl_values.std():.3f}")

    return {
        "mae_mean": mae_values.mean(),
        "mae_std": mae_values.std(),
        "mse_mean": mse_values.mean(),
        "mse_std": mse_values.std(),
        "timing_kl_mean": timing_kl_values.mean(),
        "timing_kl_std": timing_kl_values.std(),
        "velocity_kl_mean": velocity_kl_values.mean(),
        "velocity_kl_std": velocity_kl_values.std(),
    }


In [4]:
df = load_test_sequences()

# Run inference (optional)
run_dequantize_on_dataset(df)

# Evaluate
results = evaluate_full_test_set(df)


Using 72 unique sequences


  0%|          | 0/72 [00:00<?, ?it/s]


RuntimeError: Error(s) in loading state_dict for DequantTransformer:
	Missing key(s) in state_dict: "pos_proj.weight", "pos_proj.bias". 

| checkpoint                        | beat_type   | MAE mean/std (ms) | MSE mean/std (ms<sup>2</sup>) | Timing KL mean/std | Velocity KL mean/std |
|-----------------------------------|------------:|------------------:|------------------------------:|-------------------:|---------------------:|
| cp_20260209225313 (Gio 100 Epochs)| beats       |        20.05/5.64 |                 668.29/342.29 |         12.88/2.90 |            7.72/4.64 |
| cp_20260209225313                 | beats+fills |        21.69/8.48 |                 803.19/613.51 |         13.53/2.98 |           10.21/5.13 |
| cp_20260208133117 (Gio 40 Epochs) | beats       |        19.99/5.62 |                 665.50/337.98 |         12.86/3.09 |            8.11/4.81 |
| cp_20260208133117                 | beats+fills |        21.83/8.21 |                 802.31/568.63 |         13.43/3.05 |           10.63/5.43 |
| cp_20260130112914 (Amon w/o p.e.) | beats       |                   |                               |                    |                      |
| cp_20260130112914                 | beats+fills |

In [ ]:
H_path = '../.data/original.midi'
Q_path = '../.data/quantized.midi'
G_path = '../.data/dequantized_40epochs.midi'

results = evaluate(H_path, Q_path, G_path)

for k, v in results.items():
    print(f"{k}: {v:.6f}")



Timing MAE (ms): 14.917015
Timing MSE (ms²): 381.914185
Timing KL: 10.631448
Velocity KL: 2.136246
